# 🚀 Phase 9 Training Notebook (Balanced Hyperparameters)

## Lessons from Previous Phases:
| Phase | Issue | Result |
|-------|-------|--------|
| **Phase 7** | 15 epochs + 1e-4 LR | Overfit (Train=0.83, Val=0.50) |
| **Phase 8** | MICRO + 1e-5 LR | Underfit (F1=0.51, low precision) |

## Phase 9 Strategy:
- **MEDIUM tier** (11.98M params) - sufficient capacity
- **LR: 5e-5** - half of Phase 7 for stability
- **Epochs: 10** with early stopping (patience=3)
- **Higher regularization** - dropout 0.3, label smoothing 0.15

## Target Metrics:
| Metric | Target |
|--------|--------|
| **F1** | **≥ 0.85** |
| **Recall** | **≥ 0.85** |
| **Precision** | **≥ 0.85** |

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone/Update Repository

In [ ]:
import os

if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
    print("✅ Repository cloned successfully!")
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content
    print("✅ Repository updated to latest!")

## Step 3: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm --quiet
print("✅ Dependencies installed!")

## Step 4: Configure Paths & Hyperparameters

In [ ]:
# ============================================
# Phase 9 Configuration - Balanced Training
# ============================================

OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output"
LABELS = "/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/all_labels.csv"

# Phase 9 output directory
OUT_DIR = "/content/drive/MyDrive/DAIC-WOZ_Datasets/checkpoints_phase9"

# Balanced hyperparameters
TIER = "medium"  # 11.98M params - sufficient capacity
EPOCHS = 10      # Moderate epochs
LR = 5e-5        # Lower LR for stability (half of Phase 7)

import os

# Clean checkpoint directory
!rm -rf {OUT_DIR}
!mkdir -p {OUT_DIR}
print(f"🧹 Cleaned and created: {OUT_DIR}")

# Verify paths
print(f"\n📁 OS_PATH exists: {os.path.exists(OS_PATH)}")
print(f"📁 DATA_DIR exists: {os.path.exists(DATA_DIR)}")
print(f"📁 LABELS exists: {os.path.exists(LABELS)}")
print(f"\n⚙️ Phase 9 Configuration:")
print(f"   Tier: {TIER} (11.98M params)")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LR}")
print(f"   Target: F1 ≥ 0.85, Recall ≥ 0.85, Precision ≥ 0.85")

## Step 5: Train All 5 Folds

In [ ]:
# ============================================
# Phase 9 Training - Balanced Hyperparameters
# ============================================

import time

total_start = time.time()

for fold in range(5):
    print(f"\n{'='*60}")
    print(f"🚀 PHASE 9 - FOLD {fold}/4 (MEDIUM tier, LR={LR})")
    print(f"{'='*60}\n")

    fold_start = time.time()

    # Train with balanced settings
    !PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/train.py \
        --data_dir {DATA_DIR} \
        --labels_csv {LABELS} \
        --output_dir {OUT_DIR} \
        --tier {TIER} \
        --epochs {EPOCHS} \
        --fold_idx {fold} \
        --lr {LR}

    fold_time = time.time() - fold_start
    print(f"\n⏱️ Fold {fold} completed in {fold_time/60:.1f} minutes")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"✅ ALL 5 FOLDS COMPLETED!")
print(f"⏱️ Total training time: {total_time/60:.1f} minutes")
print(f"{'='*60}")

## Step 6: Verify Checkpoints

In [ ]:
import os
import glob

print("📦 Phase 9 Checkpoints:")
checkpoints = sorted(glob.glob(f"{OUT_DIR}/*_best.pt"))

if not checkpoints:
    print("   ⚠️ No checkpoints found!")
else:
    for ckpt in checkpoints:
        size_mb = os.path.getsize(ckpt) / (1024*1024)
        print(f"   ✅ {os.path.basename(ckpt)} ({size_mb:.1f} MB)")
    print(f"\n✅ Total: {len(checkpoints)} checkpoints")

## Step 7: Run Ensemble Prediction

In [ ]:
print("🔮 Generating ensemble predictions...\n")

!PYTHONPATH={OS_PATH} python {OS_PATH}/scripts/ensemble_predict.py \
    --checkpoints {OUT_DIR} \
    --input {DATA_DIR} \
    --tier {TIER} \
    --output "/content/phase9_results.csv"

import os
if os.path.exists("/content/phase9_results.csv"):
    print("\n✅ Ensemble predictions saved!")
else:
    print("\n❌ Ensemble prediction failed!")

## Step 8: Evaluate & Optimize Threshold

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

print("🏆 Phase 9 Results Analysis...")

# Load results and labels
results_df = pd.read_csv("/content/phase9_results.csv")
labels_df = pd.read_csv(LABELS)

# Merge
results_df['pid'] = results_df['pid'].astype(str)
labels_df['Participant_ID'] = labels_df['Participant_ID'].astype(str)
merged = results_df.merge(labels_df, left_on='pid', right_on='Participant_ID', how='inner')

y_prob = merged['probability'].values
if 'PHQ8_Binary' in merged.columns:
    y_true = merged['PHQ8_Binary'].values
else:
    y_true = (merged['PHQ8_Score'] >= 10).astype(int).values

# Find optimal threshold
best_f1, best_thresh = 0, 0.45
for thresh in np.arange(0.30, 0.70, 0.01):
    y_pred = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_true, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

# Final metrics at optimal threshold
y_pred_opt = (y_prob >= best_thresh).astype(int)
f1 = f1_score(y_true, y_pred_opt)
precision = precision_score(y_true, y_pred_opt)
recall = recall_score(y_true, y_pred_opt)
accuracy = accuracy_score(y_true, y_pred_opt)
cm = confusion_matrix(y_true, y_pred_opt)

print(f"\n{'='*50}")
print(f"📊 PHASE 9 FINAL RESULTS (Threshold={best_thresh:.2f})")
print(f"{'='*50}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Confusion Matrix:")
print(cm)
print(f"{'='*50}")

# Target check
print("\n🎯 Target Check (≥ 0.85):")
print(f"  {'✅' if f1 >= 0.85 else '❌'} F1: {f1:.4f}")
print(f"  {'✅' if recall >= 0.85 else '❌'} Recall: {recall:.4f}")
print(f"  {'✅' if precision >= 0.85 else '❌'} Precision: {precision:.4f}")

## Step 9: Save Results

In [ ]:
!cp /content/phase9_results.csv "/content/drive/MyDrive/DAIC-WOZ_Datasets/phase9_final_results.csv"
print("✅ Results saved to Google Drive!")

print(f"\n{'='*50}")
print(f"🏆 PHASE 9 COMPLETE")
print(f"{'='*50}")
print(f"📁 Checkpoints: {OUT_DIR}")
print(f"🎯 Best F1: {best_f1:.4f}")
print(f"{'='*50}")